# Sentence Embeddings mit Sentence Transformers

In diesem Notebook lernst du, wie Sprachmodelle Sätze als **Zahlenvektor** (sog. *Embedding*) darstellen und wie damit die **semantische Ähnlichkeit** zwischen Sätzen gemessen werden kann.

**Lernziele:**
- Du verstehst, was ein Embedding ist und warum es nützlich ist.
- Du kannst ein vortrainiertes Modell verwenden, um Sätze in Embeddings umzuwandeln.
- Du kannst die Ähnlichkeit zwischen Sätzen berechnen und die Ergebnisse interpretieren.

Zunächst installieren wir die benötigten Bibliotheken:

In [ ]:
pip install torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
pip install sentence-transformers

## Was ist ein Sentence Embedding?

Ein **Embedding** ist eine Darstellung eines Satzes als Liste von Zahlen (ein sogenannter *Vektor*). Dabei gilt: **Sätze mit ähnlicher Bedeutung liegen im Zahlenraum nahe beieinander.**

Stell dir vor, jeder Satz bekommt eine „Adresse" in einem hochdimensionalen Raum. Sätze, die inhaltlich ähnlich sind, haben ähnliche Adressen — egal ob sie dieselben Wörter verwenden oder nicht.

Als Nächstes importieren wir das Sentence-Transformer-Modell:

In [ ]:
from sentence_transformers import SentenceTransformer  # Import der Klasse SentenceTransformer aus dem Modul sentence_transformers

## Das Bank-Beispiel: Polysemie

Das Wort **„Bank"** hat im Deutschen zwei völlig unterschiedliche Bedeutungen:

- Eine **Sitzgelegenheit** (z. B. im Park)
- Ein **Geldinstitut** (z. B. für Überweisungen)

Wir verwenden drei Sätze, um zu testen, ob das Modell diesen Unterschied „versteht":

> **Satz 1:** Ich gehe spazieren und setze mich auf eine Bank. *(Parkbank)*  
> **Satz 2:** Ich gehe zur Bank, um Geld abzuheben. *(Geldinstitut)*  
> **Satz 3:** Ich muss eine Überweisung abgeben, um meine Rechnung zu bezahlen. *(Thema: Finanzen – kein „Bank" im Satz!)*

**Hypothese:** Satz 2 und Satz 3 sollten ähnlicher sein als Satz 1 und Satz 3 — obwohl Satz 3 das Wort „Bank" gar nicht enthält.

Jetzt erzeugen wir die Embeddings:

In [ ]:
sentences = [
    "Ich gehe spazieren und setze mich auf eine Bank.",                   # Parkbank
    "Ich gehe zur Bank, um Geld abzuheben.",                             # Geldinstitut
    "Ich muss eine Überweisung abgeben, um meine Rechnung zu bezahlen."  # Finanzen (kein "Bank" im Satz)
]

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(sentences)

# Nur die ersten 5 Werte des ersten Embeddings anzeigen – der volle Vektor hat 384 Dimensionen
print(f"Embedding von Satz 1 (erste 5 von {len(embeddings[0])} Werten):")
print(embeddings[0][:5])

## Die Dimensionen der Embeddings

Jeder Satz wird als Vektor mit **384 Dimensionen** dargestellt — das Modell kodiert also die Bedeutung eines Satzes in 384 Zahlen.

> Zum Vergleich: Ein Pixel in einem Farbbild hat 3 Dimensionen (Rot, Grün, Blau). Ein Satzembedding hat 384 Zahlen, die gemeinsam die *semantische Bedeutung* des Satzes codieren.

Lass uns die Dimension aller drei Sätze überprüfen:

In [ ]:
for i, satz in enumerate(sentences):
    print(f"Satz {i+1}: \"{satz}\"")
    print(f"  → Dimension: {len(embeddings[i])}\n")

## Ähnlichkeit messen: Die Kosinus-Ähnlichkeit

Um zu vergleichen, wie ähnlich sich zwei Embeddings sind, verwenden wir die **Kosinus-Ähnlichkeit** (*cosine similarity*).

Der Wert liegt immer zwischen **-1 und 1**:

| Wert | Bedeutung |
|------|-----------|
| **1** | Die Sätze sind inhaltlich identisch |
| **~0,8** | Die Sätze sind sehr ähnlich |
| **~0,5** | Die Sätze haben einen losen inhaltlichen Zusammenhang |
| **0** | Kein inhaltlicher Zusammenhang |
| **-1** | Die Sätze bedeuten das genaue Gegenteil |

Jetzt vergleichen wir alle drei Satzpaare:

In [ ]:
print(f"Satz 1: \"{sentences[0]}\"")
print(f"Satz 2: \"{sentences[1]}\"")
ähnlichkeit = model.similarity(embeddings[0], embeddings[1]).item()
print(f"→ Ähnlichkeit: {ähnlichkeit:.4f}\n")

In [ ]:
print(f"Satz 2: \"{sentences[1]}\"")
print(f"Satz 3: \"{sentences[2]}\"")
ähnlichkeit = model.similarity(embeddings[1], embeddings[2]).item()
print(f"→ Ähnlichkeit: {ähnlichkeit:.4f}\n")

In [ ]:
print(f"Satz 1: \"{sentences[0]}\"")
print(f"Satz 3: \"{sentences[2]}\"")
ähnlichkeit = model.similarity(embeddings[0], embeddings[2]).item()
print(f"→ Ähnlichkeit: {ähnlichkeit:.4f}\n")

## Reflexion und Aufgaben

### Ergebnisse interpretieren
Schau dir die drei Ähnlichkeitswerte an und beantworte folgende Fragen:

- **Satz 1 & Satz 2** (Parkbank vs. Geldinstitut): War der Wert hoch oder niedrig? Warum?
- **Satz 2 & Satz 3** (Geldinstitut vs. Überweisung): Was fällt auf, obwohl Satz 3 das Wort „Bank" gar nicht enthält?
- **Satz 1 & Satz 3** (Parkbank vs. Überweisung): Macht das Ergebnis Sinn?

---

### Aufgaben

**Aufgabe 1 – Eigene Sätze testen:**  
Ergänze die Liste `sentences` um einen vierten Satz, der thematisch zur Finanzwelt gehört (z. B. *„Ich spare jeden Monat einen Teil meines Gehalts."*). Berechne danach die Ähnlichkeit zwischen deinem neuen Satz und Satz 1.

**Aufgabe 2 – Hypothese prüfen:**  
Formuliere zunächst eine Hypothese: Welche beiden Sätze werden deiner Meinung nach am ähnlichsten sein? Überprüfe anschließend deine Vermutung mit dem Code.

**Aufgabe 3 – Synonyme testen:**  
Ersetze Satz 2 durch einen bedeutungsgleichen Satz *ohne* das Wort „Bank", z. B.:  
*„Ich besuche das Geldinstitut, um Bargeld zu bekommen."*  
Wie ändert sich die Ähnlichkeit zu Satz 1? Was sagt das über das Modell aus?

**Aufgabe 4 (Erweiterung) – Alle Paare auf einmal:**  
Kannst du mit einer verschachtelten `for`-Schleife alle möglichen Satzpaare automatisch vergleichen, ohne jeden Vergleich einzeln aufzuschreiben?